# ETO-Core: Evolutionary Transition Operator

Training pipeline for three model variants:
- **Model A (Static)**: Sequence-only baseline
- **Model B (History)**: Sequence + history baseline  
- **Model C (ETO-Core)**: Full model with transition operator

All results are saved to Google Drive for persistence.

## 1. Environment Setup

In [2]:
import os
import sys
import json
import time
import numpy as np
import torch

print('=' * 60)
print('ETO-Core Training Pipeline')
print('=' * 60)
print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')
    device = 'cuda'
else:
    print('WARNING: No GPU detected. Training will be very slow.')
    print('Go to Runtime > Change runtime type > GPU')
    device = 'cpu'

ETO-Core Training Pipeline
Python: 3.13.15
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4 (15.6 GB)


## 2. Mount Google Drive

In [4]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/ETO'
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/data', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/experiments', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/src', exist_ok=True)
print(f'Drive mounted. Working directory: {DRIVE_ROOT}')

Mounted at /content/drive
Drive mounted. Working directory: /content/drive/MyDrive/ETO


## 3. Clone / Update Repository

In [5]:
%%bash
cd /content
if [ ! -d 'ETO' ]; then
    git clone https://github.com/jianghu105/ETO.git
else
    cd ETO && git pull
fi
cd /content/ETO/ETO
pip install -q -r requirements.txt 2>/dev/null || true

Cloning into 'ETO'...
bash: line 7: cd: /content/ETO/ETO: No such file or directory


## 4. Check Data Availability

In [7]:
DATA_PATH = '/content/ETO/dataset/20230531AptDMC双端-MPO.xlsx'

if os.path.exists(DATA_PATH):
    size_mb = os.path.getsize(DATA_PATH) / 1e6
    print(f'Data found: {DATA_PATH} ({size_mb:.1f} MB)')
else:
    print(f'ERROR: Data not found at {DATA_PATH}')
    print('Make sure the dataset is in the repository.')
    raise FileNotFoundError(DATA_PATH)

Data found: /content/ETO/dataset/20230531AptDMC双端-MPO.xlsx (30.4 MB)


## 5. Configuration

In [10]:
OUTPUT_DIR = f'{DRIVE_ROOT}/experiments/etocore_v1'
SEED = 42
EPOCHS = 100
SMOKE_TEST = False  # Set True for quick validation

if SMOKE_TEST:
    EPOCHS = 5
    OUTPUT_DIR = f'{DRIVE_ROOT}/experiments/smoke_test'
    print('SMOKE TEST MODE: 5 epochs only')

gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
BATCH_SIZE = 512 if gpu_mem_gb >= 14 else (256 if gpu_mem_gb >= 7 else 128)

config = {
    'seed': SEED,
    'batch_size': BATCH_SIZE,
    'training': {
        'lr': 1e-3,
        'weight_decay': 1e-4,
        'max_grad_norm': 1.0,
        'epochs': EPOCHS,
        'patience': 15,
        'huber_delta': 1.0,
        'use_amp': torch.cuda.is_available(),
    },
}

print(f'Output: {OUTPUT_DIR}')
print(f'Batch size: {BATCH_SIZE}')
print(f'Epochs: {EPOCHS}')

Output: /content/drive/MyDrive/ETO/experiments/etocore_v1
Batch size: 512
Epochs: 100


## 6. Smoke Test (Quick Validation)

In [12]:
if SMOKE_TEST:
    sys.path.insert(0, '/content/ETO')

    from data.unified_dataset import build_etocore_dataset
    from data.eto_dataset import build_dataloaders
    from models.eto_core import build_model
    from training.eto_core_trainer import Trainer

    print('Running smoke test...')
    dataset_dict = build_etocore_dataset(DATA_PATH, seed=SEED)
    loaders = build_dataloaders(dataset_dict, batch_size=BATCH_SIZE)

    model = build_model('etocore')
    trainer = Trainer(model, config, device=device)

    batch = next(iter(loaders['train']))
    batch = {k: v.to(device) for k, v in batch.items()}
    out = model(batch)

    print(f'Forward pass OK: prediction shape = {out["prediction"].shape}')

    from training.eto_core_trainer import MaskedHuberLoss
    criterion = MaskedHuberLoss()
    loss = criterion(out['prediction'], batch['target'], batch['target_mask'])
    loss.backward()
    print(f'Backward pass OK: loss = {loss.item():.4f}')
    print('Smoke test PASSED')

## 7. Load Data and Build Splits

In [13]:
sys.path.insert(0, '/content/ETO')

from data.unified_dataset import build_etocore_dataset
from data.eto_dataset import build_dataloaders

dataset_dict = build_etocore_dataset(
    data_path=DATA_PATH,
    seq_length=36,
    min_total_count=10,
    train_frac=0.7,
    val_frac=0.15,
    seed=SEED,
)

metadata = dataset_dict['metadata']
print(f"\nData summary:")
print(f"  Sequences: {metadata['n_sequences']}")
print(f"  Families: {metadata['n_families']}")
print(f"  Total samples: {metadata['total_samples']}")
print(f"  Transitions: {metadata['valid_transitions']}")

Loading data from /content/ETO/dataset/20230531AptDMC双端-MPO.xlsx
Detected sequence column: '序列/随机区长度36'
Detected round columns: {'R3': 'R3', 'R5': 'R5', 'R9': 'R9', 'R10': 'R10', 'R11': 'R11'}
Raw rows: 1048575
After cleaning: 15630 valid 36nt sequences
Assigning family IDs (k=8, min_overlap=0.3)
  2078 families from 15630 sequences
Building transition samples
  46890 total samples (15630 seqs × 3 transitions)
Splitting by family
  train: 35808 samples, 11936 seqs, 1454 families
  val: 5604 samples, 1868 seqs, 311 families
  test: 3652 samples, 1826 seqs, 313 families
  test_ood_temporal: 1826 samples, 1826 seqs, 313 families
Fitting target normalizer from training data
  mean=1.5845, std=1.7662
  Family split verified: zero overlap between splits

Data summary:
  Sequences: 15630
  Families: 2078
  Total samples: 46890
  Transitions: [('R3', 'R5'), ('R5', 'R9'), ('R9', 'R10')]


## 8. Train All Models

In [14]:
from models.eto_core import build_model
from training.eto_core_trainer import Trainer, evaluate_on_test

os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(f'{OUTPUT_DIR}/environment.json', 'w') as f:
    json.dump({
        'python': sys.version.split()[0],
        'torch': torch.__version__,
        'cuda': torch.cuda.is_available(),
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        'config': config,
        'metadata': metadata,
    }, f, indent=2, default=str)

all_results = {}
for model_type in ['static', 'history', 'etocore']:
    model_dir = f'{OUTPUT_DIR}/{model_type}'
    os.makedirs(model_dir, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"Training {model_type.upper()}")
    print(f"{'='*60}")

    torch.manual_seed(SEED)
    np.random.seed(SEED)

    loaders = build_dataloaders(dataset_dict, batch_size=BATCH_SIZE)
    model = build_model(model_type)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Parameters: {n_params:,}")

    trainer = Trainer(model, config, device=device)

    ckpt_path = f'{model_dir}/checkpoint_last.pt'
    if os.path.exists(ckpt_path):
        print(f"  Resuming from checkpoint...")
        trainer.load_checkpoint(ckpt_path)

    t0 = time.time()
    train_result = trainer.train(loaders['train'], loaders['val'], model_dir)
    train_time = time.time() - t0

    print(f"\n  Best epoch: {train_result['best_epoch']}, "
          f"val_loss: {train_result['best_val_loss']:.4f}, "
          f"time: {train_time:.0f}s")

    test_results = {}
    if 'test' in loaders:
        test_results = evaluate_on_test(
            model, loaders['test'], dataset_dict['normalizer'], device)
        print(f"  Test Spearman: {test_results['overall']['spearman']:.4f}")
        print(f"  Test NDCG@50:  {test_results['overall']['ndcg@50']:.4f}")

    ood_results = {}
    if 'test_ood_temporal' in loaders and len(loaders['test_ood_temporal'].dataset) > 0:
        ood_results = evaluate_on_test(
            model, loaders['test_ood_temporal'], dataset_dict['normalizer'], device)
        print(f"  OOD Spearman: {ood_results['overall']['spearman']:.4f}")

    results = {
        'model_type': model_type,
        'n_params': n_params,
        'train_time_s': train_time,
        'best_epoch': train_result['best_epoch'],
        'best_val_loss': train_result['best_val_loss'],
        'test': {
            'overall': test_results.get('overall', {}),
            'per_transition': test_results.get('per_transition', {}),
        },
        'ood_temporal': {'overall': ood_results.get('overall', {})},
        'seed': SEED,
    }

    with open(f'{model_dir}/results.json', 'w') as f:
        json.dump(results, f, indent=2, default=str)

    all_results[model_type] = results


Training STATIC
  Parameters: 76,161
  Epoch   0 | train_loss=0.4105 val_loss=0.3899 val_spearman=0.0351 val_ndcg=0.0510 * best
  Epoch   1 | train_loss=0.4074 val_loss=0.3876 val_spearman=0.0344 val_ndcg=0.0617 * best
  Epoch   8 | train_loss=0.4065 val_loss=0.3873 val_spearman=0.0326 val_ndcg=0.0467 * best
  Epoch  10 | train_loss=0.4050 val_loss=0.3875 val_spearman=0.0352 val_ndcg=0.0197 
  Epoch  11 | train_loss=0.4047 val_loss=0.3872 val_spearman=0.0390 val_ndcg=0.0329 * best
  Epoch  12 | train_loss=0.4044 val_loss=0.3872 val_spearman=0.0394 val_ndcg=0.0235 * best
  Epoch  13 | train_loss=0.4052 val_loss=0.3869 val_spearman=0.0403 val_ndcg=0.0264 * best
  Epoch  18 | train_loss=0.4037 val_loss=0.3867 val_spearman=0.0487 val_ndcg=0.0325 * best
  Epoch  20 | train_loss=0.4036 val_loss=0.3881 val_spearman=0.0446 val_ndcg=0.0344 
  Epoch  22 | train_loss=0.4041 val_loss=0.3865 val_spearman=0.0517 val_ndcg=0.0313 * best
  Epoch  30 | train_loss=0.4031 val_loss=0.3874 val_spearman=0.0

## 9. Results Summary

In [15]:
with open(f'{OUTPUT_DIR}/all_results.json', 'w') as f:
    json.dump(all_results, f, indent=2, default=str)

print('=' * 70)
print('FINAL COMPARISON')
print('=' * 70)
print(f"{'Model':<12} {'Spearman':>10} {'Pearson':>10} {'NDCG@50':>10} "
      f"{'MAE':>10} {'Params':>10}")
print('-' * 70)

for mt, r in all_results.items():
    test = r.get('test', {}).get('overall', {})
    print(f"{mt:<12} "
          f"{test.get('spearman', 0):>10.4f} "
          f"{test.get('pearson', 0):>10.4f} "
          f"{test.get('ndcg@50', 0):>10.4f} "
          f"{test.get('mae', 0):>10.4f} "
          f"{r['n_params']:>10,}")

print()
print('Per-transition breakdown:')
for mt, r in all_results.items():
    print(f"\n  {mt.upper()}:")
    for t_name, t_m in r.get('test', {}).get('per_transition', {}).items():
        print(f"    {t_name}: Spearman={t_m['spearman']:.4f}, "
              f"NDCG@50={t_m['ndcg@50']:.4f}")
    ood = r.get('ood_temporal', {}).get('overall', {})
    if ood:
        print(f"    OOD (R9→R10): Spearman={ood.get('spearman', 0):.4f}")

print(f"\nAll results saved to: {OUTPUT_DIR}/")

FINAL COMPARISON
Model          Spearman    Pearson    NDCG@50        MAE     Params
----------------------------------------------------------------------
static           0.1109     0.1302     0.1163     1.2246     76,161
history          0.8185     0.9235     0.9731     0.2851    141,049
etocore          0.8170     0.9226     0.9829     0.2845    219,769

Per-transition breakdown:

  STATIC:
    R3_R5: Spearman=0.2522, NDCG@50=0.1879
    R5_R9: Spearman=0.0232, NDCG@50=0.0000
    OOD (R9→R10): Spearman=-0.0865

  HISTORY:
    R3_R5: Spearman=0.7641, NDCG@50=0.9731
    R5_R9: Spearman=0.0771, NDCG@50=0.4141
    OOD (R9→R10): Spearman=0.7295

  ETOCORE:
    R3_R5: Spearman=0.7662, NDCG@50=0.9829
    R5_R9: Spearman=-0.0193, NDCG@50=0.3465
    OOD (R9→R10): Spearman=0.7267

All results saved to: /content/drive/MyDrive/ETO/experiments/etocore_v1/


## 10. Key Questions

Based on the results above, answer:

1. **Does history provide value over static sequence features?**
   - Compare Static vs History Spearman on test set

2. **Does explicit transition provide value over direct prediction?**
   - Compare History vs ETO-Core Spearman on test set

3. **Does the model generalize to unseen families?**
   - Check test Spearman (family-level held-out)

4. **Does the model generalize to unseen transitions?**
   - Check OOD temporal Spearman (R9→R10 held-out)